# Model Leakage Demonstration

We will run the same XGBoost pipeline, but include the "terminmonths" feature this time.

In [99]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,)

In [100]:
df = pd.read_csv("sba_data.csv")
print(df.shape)
df.head()

(520019, 19)


,borrstate,grossapproval,approvalfy,initialinterestrate,fixedorvariableinterestind,terminmonths,projectstate,businesstype,businessage,loanstatus,revolverstatus,jobssupported,collateralind,default,is_franchise,naics_2,is_startup,state_mismatch,borrstate_freq
0,NY,5000.0,2009,7.50,V,144,NY,INDIVIDUAL,"Existing, 5 or more years",P I F,False,5.0,True,0,0,Construction,0,False,0.084322
1,NY,25000.0,2009,7.00,V,84,NY,CORPORATION,Less than 3 years old but at least 2,P I F,True,6.0,False,0,0,Construction,0,False,0.084322
2,CO,495000.0,2009,7.36,V,234,CO,CORPORATION,"Existing, 5 or more years",P I F,False,15.0,True,0,0,Retail,0,False,0.060603
3,MI,25000.0,2009,9.38,F,60,MI,INDIVIDUAL,"Existing, 5 or more years",P I F,False,0.0,False,0,0,Retail,0,False,0.054138
4,AK,12500.0,2009,9.50,V,120,AK,INDIVIDUAL,Less than 5 years old but at least 4,P I F,False,8.0,False,0,0,Retail,0,False,0.057051


In [101]:
numerical = ["grossapproval", "borrstate_freq", "initialinterestrate", "revolverstatus",
            "jobssupported", "is_franchise", "is_startup", "state_mismatch", "terminmonths"]
categorical = ["naics_2", "businesstype", "fixedorvariableinterestind"]
target = "default"

all_features = numerical + categorical
print(f"Total features: {len(all_features)}")

Total features: 12


In [102]:
train = df[df["approvalfy"] <= 2018]
test = df[df["approvalfy"] > 2018]

X_train = train[all_features]
y_train = train[target]
X_test = test[all_features]
y_test = test[target]

print(f"Train: {X_train.shape} {(X_train.shape[0])/ (X_train.shape[0] + X_test.shape[0]):.2f}%, Default rate: {y_train.mean():.2%}")
print(f"Test: {X_test.shape} {(X_test.shape[0])/ (X_train.shape[0] + X_test.shape[0]):.2f}%, Default rate: {y_test.mean():.2%}")

Train: (422156, 12) 0.81%, Default rate: 7.46%
Test: (97863, 12) 0.19%, Default rate: 8.46%


In [103]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# class imbalance weight
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos:.2f}")

scale_pos_weight: 12.40


In [104]:
from sklearn.model_selection import TunedThresholdClassifierCV
final_model = XGBClassifier(n_estimators=500, scale_pos_weight=scale_pos, eval_metric="aucpr", n_jobs=-1, random_state=310,
                        max_depth=5, min_child_weight=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8)

threshold_tuner = TunedThresholdClassifierCV(estimator=final_model, scoring="average_precision", cv=5)
threshold_tuner.fit(X_train, y_train)

print(f"Optimal threshold: {threshold_tuner.best_threshold_}")

Optimal threshold: 0.8276280164718628


## Results

We can see the leaky model jumped the ROC-AUC from 0.74 to 0.94, and the PR-AUC from 0.22 to 0.72. These are respectable metrics. This looks like a great model.

In [105]:
final_model.fit(X_train, y_train)
prob = final_model.predict_proba(X_test)[:, 1]
pred = (prob >= threshold_tuner.best_threshold_).astype(int)
print("___LEAKAGE___")
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall: {recall_score(y_test, pred):.4f}")
print(f"F1: {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, prob):.4f}")

___LEAKAGE___
Accuracy: 0.9505
Precision: 0.7085
Recall: 0.7048
F1: 0.7066
ROC-AUC:  0.9402
PR-AUC:  0.7211


## Feature importance

However, we can see terminmonths dominating the feature importance at 0.35, being 4 times larger than the next important feature. This model has learned to identify loans by mostly looking at unusual term lengths. The model is not predicting default. It is reading the answer off the label. This is what leakage looks like in a real dataset. It's not an obvious mistake, but a plausible-seeming feature that contains outcome information encoded in a non-obvious way. 

In [106]:
importance_df = pd.DataFrame({
    'Feature': preprocessor.get_feature_names_out(),
    'Importance': final_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance_df)

                               Feature  Importance
8                    num__terminmonths    0.350901
7                  num__state_mismatch    0.073857
32   cat__fixedorvariableinterestind_F    0.064735
33   cat__fixedorvariableinterestind_V    0.060782
3                  num__revolverstatus    0.054021
2             num__initialinterestrate    0.041412
10            cat__naics_2_Agriculture    0.030127
0                   num__grossapproval    0.023596
5                    num__is_franchise    0.023391
1                  num__borrstate_freq    0.018567
16            cat__naics_2_Health Care    0.016029
15   cat__naics_2_Food & Accommodation    0.015284
25                 cat__naics_2_Retail    0.015086
6                      num__is_startup    0.013927
24            cat__naics_2_Real Estate    0.013640
31       cat__businesstype_PARTNERSHIP    0.013633
26         cat__naics_2_Transportation    0.013340
22  cat__naics_2_Professional Services    0.011700
20                 cat__naics_2